# 🔬 CourseWork Deepfake Benchmark — Chuyên Sâu EDA & Phân Tích Lỗi (Deep Error Analysis)
## Khám Phá Chi Tiết Kết Quả DINOv3 ViT, Các Trường Hợp Khó Phát Hiện (% Thấp), Trực Quan Hóa Toàn Bộ Biểu Đồ & Mẫu Ảnh Sai Lệch

Notebook này thực hiện phân tích toàn diện (Forensic Deep Dive) về hiệu năng của mô hình **DINOv3 ViT-Small/16 (Finetune v3)** trên bộ **Test CourseWork Chuẩn 44 Methods (21,446 ảnh - 100% Zero-Leakage)**.

### 🎯 Nội Dung Chính:
1. **Tổng quan Hiệu năng & Thống kê Toàn cục:** Ma trận nhầm lẫn (Confusion Matrix), ROC Curve, Precision-Recall Curve, Phân phối xác suất (KDE).
2. **Phân loại Theo Nhóm Công Nghệ Sinh Ảnh (Category Breakdown):** So sánh GAN vs Diffusion vs Face Swap vs Face Reenactment vs Audio-Driven vs Real.
3. **Phân Tích Sâu Các Trường Hợp Điểm Thấp / Khó Phát Hiện:** Giải mã chi tiết tại sao `deepfake_faceswap` (77%), `lia` (87%), `facedancer` (92%) lại gặp khó khăn và phân tích độ nhạy ngưỡng (Threshold Sweep).
4. **Phòng Trưng Bày Mẫu Ảnh Thực Tế (Visual Error Gallery):** Trực quan hóa các bức ảnh **False Negatives (Fake bị lọt lưới)**, **False Positives (Real bị nghi ngờ)** và **True Positives (Bắt trúng Fake khó)**.


## 0. Setup Môi Trường, Thư Viện & Nạp Dữ Liệu Dự Đoán


In [ ]:
# ============================================================
# Setup & Imports
# ============================================================
import os, sys, csv, json, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import seaborn as sns
    has_seaborn = True
except ImportError:
    has_seaborn = False

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix, average_precision_score)

# Cấu hình thẩm mỹ biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'figure.titlesize': 14
})

# Đường dẫn dữ liệu
HT = Path("/workspace/hoangtuan/deepfake-ViT")
BENCH_DIR = Path("/workspace/data/zero_leakage_benchmark_fixed")
OUT_DIR = HT / "experiments/results/courseWorkCheck"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_BAL_CSV = BENCH_DIR / "test_coursework_44methods_balanced_zero_leakage.csv"
NPZ_PATH = OUT_DIR / "v3_pred_coursework_44methods_balanced_21k.npz"

# Đọc dữ liệu và predictions
df = pd.read_csv(TEST_BAL_CSV)
d = np.load(NPZ_PATH)
df['pred'] = d['preds']
df['prob_fake'] = d['probs']
df['prob_real'] = 1.0 - df['prob_fake']
df['correct'] = df['label'] == df['pred']

# Gán nhóm danh mục công nghệ (Category)
def categorize_method(m):
    m = m.lower()
    if 'real' in m or 'ffhq' in m:
        return 'Real Faces'
    elif any(k in m for k in ['swap', 'dancer', 'inswap', 'simswap', 'blendface']):
        return 'Face Swap / Blending'
    elif any(k in m for k in ['stylegan', 'vqgan', 'stargan', 'e4e', 'e4s', 'styleclip', 'whichfaceisreal']):
        return 'GAN Generators'
    elif any(k in m for k in ['dit', 'sit', 'sd2.1', 'pixart', 'rddm', 'ddim', 'midjourney']):
        return 'Diffusion / Text2Img'
    elif any(k in m for k in ['sadtalker', 'wav2lip']):
        return 'Audio-Driven / Talking Head'
    elif any(k in m for k in ['fomm', 'mraa', 'lia', 'mcnet', 'tpsm', 'facevid2vid', 'hyperreenact', 'pirender', 'one_shot', 'danet', 'uniface', 'fsgan', 'heygen', 'deepfacelab']):
        return 'Face Reenactment / Puppeteering'
    return 'Other Deepfakes'

df['category'] = df['method'].apply(categorize_method)

print(f"✅ Đã nạp thành công: {len(df):,} mẫu test từ {TEST_BAL_CSV.name}")
print(f"   - Số lượng phương pháp: {df['method'].nunique()} methods")
print(f"   - Phân bổ: {len(df[df['label']==0]):,} Real : {len(df[df['label']==1]):,} Fake (Cân bằng 1:1)")


---
# Section 1 — Tổng Quan Hiệu Năng & Các Biểu Đồ Chuẩn Toàn Cục

## 1.1 Ma Trận Nhầm Lẫn (Confusion Matrix) — Số Lượng & Tỷ Lệ Chuẩn Hóa


In [ ]:
# Tính toán các chỉ số toàn cục
labels = df['label'].values
preds = df['pred'].values
probs = df['prob_fake'].values

acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs)
f1 = f1_score(labels, preds)
prec = precision_score(labels, preds)
rec = recall_score(labels, preds)
cm = confusion_matrix(labels, preds, labels=[0, 1])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Raw Confusion Matrix
im1 = axes[0].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=16, fontweight='bold')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Real (0)", "Fake (1)"], fontsize=11, fontweight='semibold')
axes[0].set_yticklabels(["Real (0)", "Fake (1)"], fontsize=11, fontweight='semibold')
axes[0].set_xlabel("Dự Đoán (Predicted)", fontsize=12)
axes[0].set_ylabel("Nhãn Thật (Ground Truth)", fontsize=12)
axes[0].set_title(f"Confusion Matrix (Số Mẫu)\nAccuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%", fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# 2. Normalized Confusion Matrix
im2 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm_norm[i,j]*100:.2f}%", ha="center", va="center",
                     color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=16, fontweight='bold')
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Real (0)", "Fake (1)"], fontsize=11, fontweight='semibold')
axes[1].set_yticklabels(["Real (0)", "Fake (1)"], fontsize=11, fontweight='semibold')
axes[1].set_xlabel("Dự Đoán (Predicted)", fontsize=12)
axes[1].set_ylabel("Nhãn Thật (Ground Truth)", fontsize=12)
axes[1].set_title(f"Normalized Confusion Matrix (%)\nReal Acc: {cm_norm[0,0]*100:.2f}% | Fake Recall: {cm_norm[1,1]*100:.2f}%", fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_confusion_matrices.png", dpi=150)
plt.show()


## 1.2 Đường Cong ROC & Đường Cong Precision-Recall (PR Curve)


In [ ]:
# ROC & PR Curves
fpr, tpr, roc_thresholds = roc_curve(labels, probs)
precision_pts, recall_pts, pr_thresholds = precision_recall_curve(labels, probs)
ap_score = average_precision_score(labels, probs)

# Tìm Youden's J statistic để tìm optimal threshold
j_scores = tpr - fpr
opt_idx = np.argmax(j_scores)
opt_th = roc_thresholds[opt_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2980b9', lw=2.5, label=f'DINOv3 ViT (AUC = {auc*100:.2f}%)')
axes[0].plot([0, 1], [0, 1], color='#7f8c8d', linestyle='--', label='Random Guess (AUC = 50.0%)')
axes[0].scatter(fpr[opt_idx], tpr[opt_idx], color='#e74c3c', s=80, zorder=5, 
                label=f"Ngưỡng Tối Ưu = {opt_th:.3f} (TPR={tpr[opt_idx]*100:.1f}%, FPR={fpr[opt_idx]*100:.1f}%)")
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (TPR / Recall)', fontsize=11)
axes[0].set_title('Receiver Operating Characteristic (ROC Curve)', fontsize=12, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall_pts, precision_pts, color='#27ae60', lw=2.5, label=f'PR Curve (AP = {ap_score*100:.2f}%)')
axes[1].axhline(y=0.5, color='#7f8c8d', linestyle='--', label='Baseline (Prevalence = 50.0%)')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curve (PR Curve)', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_roc_pr_curves.png", dpi=150)
plt.show()


## 1.3 Phân Phối Xác Suất Dự Đoán (Probability Density & Confidence Histograms)


In [ ]:
# Biểu đồ phân phối xác suất dự đoán (Real vs Fake)
real_probs = df[df['label'] == 0]['prob_fake']
fake_probs = df[df['label'] == 1]['prob_fake']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(real_probs, bins=50, alpha=0.6, color='#27ae60', label=f'Real Faces (Mean P(Fake) = {real_probs.mean()*100:.1f}%)', density=True)
ax.hist(fake_probs, bins=50, alpha=0.6, color='#c0392b', label=f'Fake Faces (Mean P(Fake) = {fake_probs.mean()*100:.1f}%)', density=True)

ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Ngưỡng Phân Loại Mặc Định (0.50)')
ax.set_xlabel('Xác Suất Dự Đoán Fake P(Fake)', fontsize=11)
ax.set_ylabel('Mật Độ Xác Suất (Density)', fontsize=11)
ax.set_title('Phân Phối Xác Suất Dự Đoán Của DINOv3 ViT (Sự Tách Biệt Rõ Rệt Giữa Real & Fake)', fontsize=12, fontweight='bold')
ax.legend(loc='upper center', fontsize=10.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_probability_distribution.png", dpi=150)
plt.show()


---
# Section 2 — Phân Tích Theo Nhóm Công Nghệ Sinh Ảnh (Category Breakdown)

So sánh sức mạnh phát hiện của mô hình trên các họ công nghệ Deepfake khác nhau:
- **Diffusion / Text2Img** (`DiT`, `SiT`, `sd2.1`, `pixart`, `RDDM`, `ddim`, `MidJourney`)
- **GAN Generators** (`StyleGAN2/3/XL`, `VQGAN`, `stargan`, `styleclip`, `e4e`, `e4s`)
- **Face Swap / Blending** (`faceswap`, `facedancer`, `blendface`, `simswap`, `inswap`, `deepfake_faceswap`)
- **Face Reenactment / Puppeteering** (`fomm`, `mraa`, `lia`, `hyperreenact`, `pirender`, `facevid2vid`, `danet`, `uniface`)
- **Audio-Driven / Talking Head** (`sadtalker`, `wav2lip`)
- **Real Faces** (Mặt Thật)


In [ ]:
# Tính toán hiệu năng theo nhóm công nghệ
cat_summary = []
for cat, grp in df.groupby('category'):
    c_acc = grp['correct'].mean() * 100
    c_auc = roc_auc_score(grp['label'], grp['prob_fake']) if grp['label'].nunique() > 1 else np.nan
    cat_summary.append({
        'Category': cat,
        'Số Mẫu': len(grp),
        'Số Mẫu Đúng': int(grp['correct'].sum()),
        'Độ Chính Xác (%)': round(c_acc, 2),
        'P(Fake) Trung Bình': f"{grp['prob_fake'].mean()*100:.2f}%"
    })

df_cat = pd.DataFrame(cat_summary).sort_values(by='Độ Chính Xác (%)', ascending=False)
display(df_cat)

# Biểu đồ so sánh giữa các họ công nghệ
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#27ae60' if acc >= 95 else '#2980b9' if acc >= 90 else '#e67e22' for acc in df_cat['Độ Chính Xác (%)']]
bars = ax.barh(df_cat['Category'], df_cat['Độ Chính Xác (%)'], color=colors, edgecolor='black', linewidth=0.5)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, f"{w:.2f}%", va='center', fontsize=10, fontweight='bold')

ax.set_xlim(70, 105)
ax.set_xlabel('Độ Chính Xác (%)', fontsize=11)
ax.set_title('Hiệu Năng Phát Hiện Theo Từng Nhóm Công Nghệ Deepfake', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_category_comparison.png", dpi=150)
plt.show()


---
# Section 3 — Phân Tích Sâu Các Phương Pháp Điểm Thấp / Khó Phát Hiện

## 3.1 Bảng Xếp Hạng Toàn Bộ Các Phương Pháp Deepfake (Từ Khó Nhất Đến Dễ Nhất)


In [ ]:
# Bảng xếp hạng độ chính xác từng method Fake
fake_df = df[df['label'] == 1]
pm_list = []
for m, grp in fake_df.groupby('method'):
    m_acc = grp['correct'].mean() * 100
    m_wrong = len(grp) - int(grp['correct'].sum())
    pm_list.append({
        'Method': m,
        'Category': grp['category'].iloc[0],
        'Số Mẫu Test': len(grp),
        'Đoán Đúng': int(grp['correct'].sum()),
        'Đoán Sai (Bị lọt)': m_wrong,
        'Độ Chính Xác (%)': round(m_acc, 2),
        'Xác Suất Fake TB': round(grp['prob_fake'].mean()*100, 2)
    })

df_pm_ranked = pd.DataFrame(pm_list).sort_values(by='Độ Chính Xác (%)', ascending=True).reset_index(drop=True)
display(df_pm_ranked)

# Biểu đồ Per-Method Accuracy toàn bộ 38 phương pháp
fig, ax = plt.subplots(figsize=(12, 11))
bar_colors = ['#c0392b' if acc < 85 else '#e67e22' if acc < 93 else '#27ae60' for acc in df_pm_ranked['Độ Chính Xác (%)']]
bars = ax.barh(df_pm_ranked['Method'], df_pm_ranked['Độ Chính Xác (%)'], color=bar_colors, edgecolor='black', linewidth=0.5)

ax.axvline(90, color='#27ae60', linestyle='--', alpha=0.7, label='Ngưỡng 90% (Rất Tốt)')
ax.axvline(80, color='#e67e22', linestyle='--', alpha=0.7, label='Ngưỡng 80%')
ax.set_xlabel('Độ Chính Xác (%)', fontsize=11)
ax.set_title('Bảng Xếp Hạng Độ Chính Xác Toàn Bộ 38 Phương Pháp Deepfake (CourseWork Benchmark)', fontsize=13, fontweight='bold')
ax.set_xlim(60, 105)
ax.legend(loc='lower right')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.6, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va='center', fontsize=8.5, fontweight='semibold')

plt.tight_layout()
plt.savefig(OUT_DIR / "eda_per_method_ranking.png", dpi=150)
plt.show()


## 3.2 Giải Mã Chi Tiết: Tại Sao `deepfake_faceswap` (77%) & `lia` (87%) Lại Có Điểm Thấp Hơn?


In [ ]:
# Phân tích sâu 2 phương pháp khó nhất: deepfake_faceswap và lia
hard_methods = ['deepfake_faceswap', 'lia', 'facedancer', 'faceswap']

print("="*80)
print("🔍 PHÂN TÍCH ĐỘ NHẠY THEO NGƯỠNG (THRESHOLD SWEEP) CHO CÁC METHOD KHÓ:")
print("="*80)

for m in hard_methods:
    sub_m = df[df['method'] == m]
    sub_real = df[df['method'] == 'real'].sample(n=len(sub_m), random_state=42)
    comb = pd.concat([sub_m, sub_real])
    auc_m = roc_auc_score(comb['label'], comb['prob_fake'])
    
    print(f"\nMethod: {m.upper()} ({len(sub_m)} mẫu) | ROC-AUC vs Real: {auc_m*100:.2f}%")
    print("  Threshold | Fake Recall | Real Acc | Balanced Acc")
    print("  " + "-"*45)
    for th in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        rec_f = (sub_m['prob_fake'] >= th).mean() * 100
        rec_r = (sub_real['prob_fake'] < th).mean() * 100
        b_acc = (rec_f + rec_r) / 2
        print(f"    {th:.2f}    |    {rec_f:>5.1f}%   |   {rec_r:>5.1f}%  |    {b_acc:>5.1f}%")


---
# Section 4 — Phòng Trưng Bày Mẫu Ảnh Thực Tế (Visual Error Gallery)

Trực quan hóa trực tiếp các bức ảnh:
1. **Top False Negatives (Ảnh Fake nhưng bị model đoán nhầm là Real)** — Tìm hiểu những đặc trưng đánh lừa thị giác.
2. **Top False Positives (Ảnh Real nhưng bị model nghi ngờ là Fake)** — Tìm hiểu nguyên nhân gây nghi ngờ giả.
3. **True Positives Điển Hình Của Các Phương Pháp Khó** — Xem cách DINOv3 bắt vết cắt ghép vi tế.


In [ ]:
# Hàm vẽ lưới ảnh trực quan
def plot_image_grid(image_records, title, cols=5):
    n = len(image_records)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.4))
    axes = np.array(axes).reshape(-1)
    
    for idx, r in enumerate(image_records):
        ax = axes[idx]
        try:
            im = Image.open(r['path']).convert('RGB')
            ax.imshow(im)
            
            # Tiêu đề màu sắc theo đúng/sai
            color = '#27ae60' if r['correct'] else '#c0392b'
            ax.set_title(f"Method: {r['method']}\nGT: {r['gt_str']} | Pred: {r['pred_str']}\nP(Fake): {r['prob_fake']*100:.1f}%", 
                         fontsize=8.5, fontweight='bold', color=color)
        except Exception as e:
            ax.text(0.5, 0.5, "Image Error", ha='center', va='center')
        ax.axis('off')
        
    # Tắt các axis thừa
    for idx in range(n, len(axes)):
        axes[idx].axis('off')
        
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# 1. Thu thập Top 10 False Negatives (Fake bị đoán nhầm là Real nhiều nhất)
fn_df = df[(df['label'] == 1) & (df['pred'] == 0)].sort_values(by='prob_fake', ascending=True).head(10)
fn_records = []
for _, r in fn_df.iterrows():
    fn_records.append({
        'path': r['path'],
        'method': r['method'],
        'gt_str': 'FAKE',
        'pred_str': 'REAL',
        'prob_fake': r['prob_fake'],
        'correct': False
    })

print("📸 HIỂN THỊ CÁC MẪU FALSE NEGATIVES (Ảnh Fake Tinh Vi Lừa Được Model):")
plot_image_grid(fn_records, "Top False Negatives (Deepfake Bị Đoán Nhầm Là Người Thật)")


In [ ]:
# 2. Thu thập Top 10 False Positives (Ảnh Người Thật nhưng bị nghi ngờ là Fake)
fp_df = df[(df['label'] == 0) & (df['pred'] == 1)].sort_values(by='prob_fake', ascending=False).head(10)
fp_records = []
for _, r in fp_df.iterrows():
    fp_records.append({
        'path': r['path'],
        'method': 'real',
        'gt_str': 'REAL',
        'pred_str': 'FAKE',
        'prob_fake': r['prob_fake'],
        'correct': False
    })

print("📸 HIỂN THỊ CÁC MẪU FALSE POSITIVES (Ảnh Thật Bị Đoán Nhầm Là Fake):")
plot_image_grid(fp_records, "Top False Positives (Người Thật Bị Nghi Ngờ Là Deepfake)")


In [ ]:
# 3. Thu thập các mẫu Fake bắt trúng xuất sắc của deepfake_faceswap và lia
tp_hard_df = df[(df['method'].isin(['deepfake_faceswap', 'lia'])) & (df['correct'] == True)].sort_values(by='prob_fake', ascending=False).head(10)
tp_records = []
for _, r in tp_hard_df.iterrows():
    tp_records.append({
        'path': r['path'],
        'method': r['method'],
        'gt_str': 'FAKE',
        'pred_str': 'FAKE',
        'prob_fake': r['prob_fake'],
        'correct': True
    })

print("📸 HIỂN THỊ CÁC MẪU TRUE POSITIVES (Model Bắt Trúng Deepfake Khó):")
plot_image_grid(tp_records, "True Positives Thành Công Trên deepfake_faceswap & lia")
